ReAct stands for:

Reasoning → The agent thinks step-by-step
Acting → The agent takes actions (like calling tools, APIs, or databases)

Instead of just giving answers, a ReAct agent:

    1 .Thinks about the problem

    2 .Decides what action to take

    3. Executes the action

    4. Observes the result

In [12]:
from langchain_nvidia import  ChatNVIDIA

LLM = ChatNVIDIA(model="meta/llama3-70b-instruct", temperature=0.7)

In [9]:
LLM.invoke("What is the capital of France?").content

'The capital of France is Paris.'

In [23]:
class Agent:
    """
    A class representing an agent that can interact with the environment
    and perform actions based on the given query.
    """

    def __init__(self, llm, system_prompt=""):
        self.llm = llm
        self.system_prompt = system_prompt
        self.message = []
        if self.system_prompt:
            self.message.append({"role": "system", "content": self.system_prompt})

    def __call__(self, query):
        """
        Allow the agent to be called directly with a query.
        """
        self.message.append({"role": "user", "content": query})
        result = self.run(self.message)
        self.message.append({"role": "assistant", "content": result})
        return result

    def run(self, query):
        """
        Execute the given query using the language model and return the text response.
        """
        response = self.llm.invoke(query)
        return response.content

In [11]:
# define a system prompt that provides instructions to the agent on how to respond to user queries
system_prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an answer.
Use Thought to describe your thought about the question you have been asked.
Use Action to run one of the actions available to you -- Then return PAUSE.
Observation is the result of the action you took. Use it to update your thoughts and actions.

Your actions are:

Calculate_total_price:
e. g. calculate_total_price: apple: 2, orange: 3
Runs a calculation to determine the total price of the given items and their quantities.

Get_fruit_price:
e. g. get_fruit_price: apple
Retrieves the price of the specified fruit.

Example session:

Question: What is the total price for 2 apples and 3 oranges?
Thought: I need to find out the price of an apple and an orange first, then I can calculate the total price.
Action: get_fruit_price: apple
PAUSE
Observation: The price of an apple is $1.
Thought: Now I know the price of an apple, I need to find out the price of an orange.
Action: get_fruit_price: orange
PAUSE
Observation: The price of an orange is $0.5.
Thought: Now I have the prices of both fruits, I can calculate the total price.
Action: calculate_total_price: apple: 2, orange: 3
PAUSE
Observation: The total price for 2 apples and 3 oranges is $3.5.
Answer: The total price for 2 apples and 3 oranges is $3.5.

"""

In [14]:
fruit_prices = {
    "apple": 1,
    "orange": 0.5,
    "grapes": 2,
    "mango": 1.5
}

In [49]:
# Normalize singular/plural fruit names used by the model.
def normalize_fruit_name(fruit):
    fruit = fruit.strip().lower()
    if fruit in fruit_prices:
        return fruit
    if fruit.endswith("es") and fruit[:-2] in fruit_prices:
        return fruit[:-2]
    if fruit.endswith("s") and fruit[:-1] in fruit_prices:
        return fruit[:-1]
    if f"{fruit}s" in fruit_prices:
        return f"{fruit}s"
    return fruit


# Function to get the price of a fruit
def get_fruit_price(fruit):
    normalized = normalize_fruit_name(fruit)
    if normalized in fruit_prices:
        return f"The price of {normalized} is ${fruit_prices[normalized]}."
    return f"Fruit not found for {fruit}."

In [44]:
# Function to calculate the total price of the given items and their quantities
def calculate_total_price(items):
    """
    Accepts either:
    - a list like ["apple: 2", "orange: 3"]
    - a string like "apple: 2, orange: 3"
    """
    if isinstance(items, str):
        items = [part.strip() for part in items.split(",") if part.strip()]

    total_price = 0
    for item in items:
        if ":" not in item:
            return f"Invalid item format: {item}. Use 'fruit: quantity'."

        fruit, quantity = item.split(":", 1)
        normalized = normalize_fruit_name(fruit)

        try:
            quantity = int(quantity.strip())
        except ValueError:
            return f"Invalid quantity for {fruit.strip()}: {quantity.strip()}"

        if normalized in fruit_prices:
            total_price += fruit_prices[normalized] * quantity
        else:
            return f"Fruit not found for {fruit.strip()}."

    return f"The total price for the given items is ${total_price}."

In [45]:
known_actions = {
    "get_fruit_price": get_fruit_price,
    "calculate_total_price": calculate_total_price}

In [37]:
import re

action_re = re.compile(r"^Action: (\w+): (.+)$")

In [50]:
def query_agent(query, max_steps=6):
    agent = Agent(LLM, system_prompt)
    user_input = query
    last_action_name = None
    last_action_result = None

    for step in range(1, max_steps + 1):
        result = agent(user_input)
        print(f"Step {step} response:\n{result}\n")

        if "Answer:" in result:
            if last_action_name == "calculate_total_price" and last_action_result:
                normalized_answer = f"Answer: {last_action_result}"
                print(f"Normalized final answer: {normalized_answer}")
                return normalized_answer
            return result

        actions = [
            action_re.match(line) for line in result.splitlines() if action_re.match(line)
        ]

        if not actions:
            print("No action found. Stopping loop.")
            return result

        action_name, action_args = actions[0].groups()
        if action_name not in known_actions:
            raise Exception(f"Unknown action: {action_name}, {action_args}")

        print(f"Executing action: {action_name} with arguments: {action_args}")
        action_result = known_actions[action_name](action_args)
        print(f"Action result: {action_result}\n")

        last_action_name = action_name
        last_action_result = action_result

        # Feed observation back so the model can continue the ReAct cycle.
        user_input = f"Observation: {action_result}"

    print("Reached max steps without final answer.")
    return result

In [48]:
query_agent("What is the total price for 2 apples and 3 oranges?")

Step 1 response:
Thought: I need to find out the price of an apple and an orange first, then I can calculate the total price.

Action: get_fruit_price: apple
PAUSE

Executing action: get_fruit_price with arguments: apple
Action result: The price of apple is $1.

Step 2 response:
Thought: Now I know the price of an apple, I need to find out the price of an orange.

Action: get_fruit_price: orange
PAUSE

Executing action: get_fruit_price with arguments: orange
Action result: The price of orange is $0.5.

Step 3 response:
Thought: Now I have the prices of both fruits, I can calculate the total price.

Action: calculate_total_price: apple: 2, orange: 3
PAUSE

Executing action: calculate_total_price with arguments: apple: 2, orange: 3
Action result: The total price for the given items is $3.5.

Step 4 response:
Answer: The total price for 2 apples and 3 oranges is $3.5.



'Answer: The total price for 2 apples and 3 oranges is $3.5.'

In [40]:
query_agent("what is the rate of an apple?")

Step 1 response:
Thought: I need to find out the price of an apple.

Action: get_fruit_price: apple
PAUSE

Executing action: get_fruit_price with arguments: apple
Action result: The price of apple is $1.

Step 2 response:
Thought: Now I know the price of an apple, I can answer the question.

Action: None
PAUSE

Answer: The rate of an apple is $1.

No action found. Stopping loop.


'Thought: Now I know the price of an apple, I can answer the question.\n\nAction: None\nPAUSE\n\nAnswer: The rate of an apple is $1.'

In [53]:
query_agent("How much I need to pay for 3 mangoes and 4 grapes and 5 apples?")

Step 1 response:
Thought: I need to find out the price of a mango, a grape, and an apple first, then I can calculate the total price.

Action: get_fruit_price: mango
PAUSE

Executing action: get_fruit_price with arguments: mango
Action result: The price of mango is $1.5.

Step 2 response:
Thought: Now I know the price of a mango, I need to find out the price of a grape.

Action: get_fruit_price: grape
PAUSE

Executing action: get_fruit_price with arguments: grape
Action result: The price of grapes is $2.

Step 3 response:
Thought: Now I know the price of a mango and a grape, I need to find out the price of an apple.

Action: get_fruit_price: apple
PAUSE

Executing action: get_fruit_price with arguments: apple
Action result: The price of apple is $1.

Step 4 response:
Thought: Now I have the prices of all the fruits, I can calculate the total price.

Action: calculate_total_price: mango: 3, grape: 4, apple: 5
PAUSE

Executing action: calculate_total_price with arguments: mango: 3, grape

'Answer: The total price for the given items is $17.5.'